# Notebook 06: pmap and Data Parallelism

## Why This Matters

Training at scale requires distributing work across multiple accelerators. JAX provides two mechanisms: `pmap` (the original multi-device transform) and `jax.sharding` (the modern, more flexible approach). Understanding both is essential for anyone doing large-scale training on TPU pods or multi-GPU clusters. DeepMind's infrastructure and many Kaggle TPU competitions use these primitives directly.


In [ ]:
%pip install -q jax jaxlib flax optax numpy

In [ ]:
import jax
import jax.numpy as jnp
from jax import lax
from flax import nnx
import optax
import numpy as np

print('Devices:', jax.devices())
print('Device count:', jax.device_count())
print('Local device count:', jax.local_device_count())
print("Ready.")

## 1. jax.pmap: One Function Replica Per Device

`pmap` (parallel map) runs a function **simultaneously on all devices**, with one replica per device. The data must be split along a leading batch axis so each device gets its shard.

Key mental model:
- **vmap**: one device, vectorize over a batch axis
- **pmap**: multiple devices, one function call per device, data sharded across devices

The function you write with `pmap` runs independently on each device. To synchronize (e.g., average gradients), you use collective operations like `lax.pmean`.


In [ ]:
n_devices = jax.local_device_count()
print(f'Running on {n_devices} device(s)')

# Basic pmap: map a function across all devices
@jax.pmap
def device_square(x):
    return x ** 2

# Input: leading axis must equal n_devices
x = jnp.arange(n_devices * 4, dtype=jnp.float32).reshape(n_devices, 4)
result = device_square(x)
print('Input shape:', x.shape)
print('Output shape:', result.shape)
print('Result:', result)

# Each device sees its own shard
@jax.pmap
def get_device_id(x):
    # lax.axis_index returns the pmap axis index (device id)
    return lax.axis_index('devices') * jnp.ones_like(x)

# Use axis_name to identify pmap axis for collectives
@jax.pmap
def get_device_id2(x):
    return jnp.ones_like(x)  # just return 1s to show sharding

shard = jnp.ones((n_devices, 4))
print('\nSharded ones:', get_device_id2(shard))

## 2. Gradient Averaging with lax.pmean

In data-parallel training, each device computes gradients on its data shard. To get the correct gradient, you must **average across devices** using `lax.pmean`. This is the distributed all-reduce operation.

`lax.pmean(x, axis_name)` computes the mean of `x` across all devices in the named pmap axis. This is a blocking collective -- all devices must reach this call before any can proceed.


In [ ]:
# Data-parallel training step with gradient averaging

def simple_net(params, x):
    W, b = params
    return jax.nn.relu(x @ W + b)

def loss_fn(params, x, y):
    pred = simple_net(params, x)
    return jnp.mean((pred - y) ** 2)

@jax.pmap
def train_step_pmap(params, x_shard, y_shard):
    # Each device computes gradients on its shard
    loss, grads = jax.value_and_grad(loss_fn)(params, x_shard, y_shard)
    # Average gradients across all devices (all-reduce)
    # Note: axis_name must match pmap's axis_name argument
    # Without axis_name in pmap decorator, we can't use pmean
    return loss, grads

# For single device demo, just show the concept
key = jax.random.PRNGKey(0)
W = jax.random.normal(key, (8, 4)) * 0.1
b = jnp.zeros(4)
params = (W, b)

# Replicate params across devices
replicated_params = jax.device_put_replicated(params, jax.devices())
print('Replicated W shape:', replicated_params[0].shape)

# Create sharded data (each device gets a different shard)
X = jax.random.normal(key, (n_devices * 16, 8))
Y = jax.random.normal(key, (n_devices * 16, 4))
X_sharded = X.reshape(n_devices, 16, 8)
Y_sharded = Y.reshape(n_devices, 16, 4)

loss_shards, grad_shards = train_step_pmap(replicated_params, X_sharded, Y_sharded)
print('Loss per device:', loss_shards)
print('Gradient W shape per device:', grad_shards[0].shape)

# Mean gradient (what you would do after pmean in multi-device setting)
mean_grads_W = grad_shards[0].mean(axis=0)  # mean across device dim
print('Mean gradient W shape:', mean_grads_W.shape)

## 3. Why pmap is Being Replaced by jax.sharding

`pmap` has limitations:
1. **Rigid data layout**: the leading axis must equal device count, computed at trace time
2. **No partial replication**: difficult to express hybrid (tensor + data) parallelism
3. **No cross-host collectives**: pmap only works within a single host

**jax.sharding** (introduced ~2022) is more flexible:
- Describe how arrays are partitioned using `NamedSharding` or `PositionalSharding`
- Works with `jit` (not a separate function) -- `jax.jit(f, in_shardings=..., out_shardings=...)`
- Supports arbitrary sharding patterns across hosts in a TPU pod
- The compiler (XLA GSPMD) automatically inserts the needed collectives


In [ ]:
# Modern JAX: jax.sharding for flexible device placement
from jax.sharding import Mesh, PartitionSpec, NamedSharding

# Create a mesh (logical device grid)
devices = np.array(jax.devices())
print(f'Available devices: {devices}')

# For 1 device: trivial mesh
mesh = Mesh(devices.reshape(n_devices), axis_names=('data',))

# Sharding spec: shard first axis across 'data' dimension
data_sharding = NamedSharding(mesh, PartitionSpec('data'))
replicated_sharding = NamedSharding(mesh, PartitionSpec())  # replicated

# Create sharded arrays
x = jnp.ones((n_devices * 8, 4))
x_sharded = jax.device_put(x, data_sharding)
print('x_sharded shape:', x_sharded.shape)
print('x_sharded sharding:', x_sharded.sharding)

# JIT with explicit sharding constraints
@jax.jit
def batched_matmul(x, W):
    return x @ W

W = jnp.ones((4, 8))
result = batched_matmul(x_sharded, W)
print('Result shape:', result.shape)
print('\nIn modern JAX, use jit + sharding annotations instead of pmap.')
print('This allows XLA to automatically insert optimal collective operations.')

## Summary

### Key Concepts

| Approach | API | Best for | Limitation |
|----------|-----|---------|------------|
| `pmap` | `@jax.pmap` | Simple data parallelism | Rigid, single-host |
| `jax.sharding` | `NamedSharding` + `jit` | Flexible, multi-host | More setup |

**Gradient averaging**: Use `lax.pmean(grads, axis_name)` inside `pmap` to average gradients. In modern JAX with sharding, the compiler inserts collectives automatically.

**Next**: `07_jax_tpu_mental_model.ipynb` -- XLA, TPU architecture, and why static shapes matter.
